<a href="https://colab.research.google.com/github/peiqi101/AI-Driven-Access-Control-Policy-Generation-Engine/blob/main/CAPSTONE_DEMO3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NIST Zero Trust Website/Login Policy Generator → Rego output (copy/paste) + OPA validation + optional Gemini drafting

Input: YAML requirements

If Gemini API key exists and you enable it: Gemini drafts rules into a strict schema

Output: Rego code printed in the notebook output (copy/paste anywhere)

Also prints: draft JSON, rego tests, OPA check/test/eval results

No persistent files are written (OPA uses only temporary files that are auto-deleted)

**cell 1**

need to install OPA. for that:
%%bash
set -e

pip -q install pyyaml pydantic==2.* rich google-genai

# Install OPA (static build, Linux)
curl -L -o /usr/local/bin/opa https://openpolicyagent.org/downloads/latest/opa_linux_amd64_static
chmod +x /usr/local/bin/opa

opa version


In [ ]:
%%bash
set -e

pip -q install pyyaml pydantic==2.* rich

# Install OPA (latest static Linux build)
curl -L -o /usr/local/bin/opa https://openpolicyagent.org/downloads/latest/opa_linux_amd64_static
chmod +x /usr/local/bin/opa

opa version


Version: 1.13.2
Build Commit: fd4fdf72d1f45ad0bd971531426eb53cf6edefec-dirty
Build Timestamp: 2026-02-18T08:58:13Z
Build Hostname: 
Go Version: go1.25.7
Platform: linux/amd64
Rego Version: v1
WebAssembly: unavailable


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    86  100    86    0     0    295      0 --:--:-- --:--:-- --:--:--   296
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 49.6M  100 49.6M    0     0  29.3M      0  0:00:01  0:00:01 --:--:-- 29.3M


**cell 2**

Configure modes and keys



Modes

OPA-only mode (default if no Gemini key): Uses YAML rules directly → compile to Rego → OPA check/test/eval

Gemini drafting mode: If key exists AND you enable it AND YAML has nl_requirements (or rules empty), Gemini generates the rules.

Keys

Set GEMINI_API_KEY (preferred) or GOOGLE_API_KEY.

**cell 3**

**Mode flags + Gemini availability**

In [ ]:
import os, re, json, yaml, subprocess, textwrap, tempfile
from rich import print as rprint

USE_GEMINI_IF_AVAILABLE = True
GEMINI_MODEL = "gemini-3-flash-preview"

def has_gemini_key() -> bool:
    return bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))

# --- UI ---
try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception:
    !pip -q install ipywidgets
    import ipywidgets as widgets
    from IPython.display import display

GEMINI_AVAILABLE = False
gemini_client = None

key_box = widgets.Password(
    description="Gemini Key:",
    placeholder="Paste Gemini API key here",
    layout=widgets.Layout(width="520px")
)

apply_btn = widgets.Button(description="Apply", button_style="success")
no_keys_btn = widgets.Button(description="No Keys", button_style="warning")
out = widgets.Output()

def _set_status(msg: str, color: str = "cyan"):
    rprint(f"[{color}]{msg}[/{color}]")

def _disable_gemini(msg: str):
    global GEMINI_AVAILABLE, gemini_client
    os.environ.pop("GEMINI_API_KEY", None)
    os.environ.pop("GOOGLE_API_KEY", None)
    GEMINI_AVAILABLE = False
    gemini_client = None
    _set_status(msg, "yellow")
    _set_status("Running OPA-only mode.", "yellow")

def _enable_gemini_with_key(key: str):
    global GEMINI_AVAILABLE, gemini_client

    # Clear existing keys
    os.environ.pop("GEMINI_API_KEY", None)
    os.environ.pop("GOOGLE_API_KEY", None)

    key = (key or "").strip()
    if not key:
        _disable_gemini("No key entered.")
        return

    try:
        from google import genai
        client = genai.Client(api_key=key)

        # Tiny validation request
        _ = client.models.generate_content(model=GEMINI_MODEL, contents="OK")

        os.environ["GEMINI_API_KEY"] = key
        gemini_client = client
        GEMINI_AVAILABLE = USE_GEMINI_IF_AVAILABLE and has_gemini_key()

        _set_status("Key accepted. Gemini enabled ✅", "green")
        _set_status(f"Gemini enabled? {GEMINI_AVAILABLE}", "cyan")

    except Exception as e:
        _disable_gemini("Wrong key (or key has no access).")
        _set_status("Debug (optional):", "dim")
        rprint(str(e))

def on_apply(_):
    out.clear_output()
    with out:
        if not USE_GEMINI_IF_AVAILABLE:
            _disable_gemini("Gemini disabled by USE_GEMINI_IF_AVAILABLE=False.")
            return
        _enable_gemini_with_key(key_box.value)

def on_no_keys(_):
    out.clear_output()
    with out:
        _disable_gemini("No Keys selected.")

apply_btn.on_click(on_apply)
no_keys_btn.on_click(on_no_keys)

display(key_box, widgets.HBox([apply_btn, no_keys_btn]), out)

# ✅ Don’t decide anything yet
with out:
    _set_status("Waiting for you: paste key and click Apply, or click No Keys.", "cyan")

Password(description='Gemini Key:', layout=Layout(width='520px'), placeholder='Paste Gemini API key here')

Output()

In [ ]:
print("GEMINI_AVAILABLE =", GEMINI_AVAILABLE)
print("Has key in env =", has_gemini_key())
print("Client exists =", gemini_client is not None)

GEMINI_AVAILABLE = True
Has key in env = True
Client exists = True


**cell 4**

Strict schema (safe) + allowed condition paths


We use a strict policy draft schema so:

Rules are always in a safe, predictable format

Gemini cannot invent random attributes

We can compile cleanly into Rego (policy-as-code)

**cell 5**

**Pydantic schema + validation**

In [ ]:
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field, model_validator

# Only allow these attribute paths inside conditions (prevents hallucinated fields)
ALLOWED_CONDITION_PATHS = {
    # Subject / identity
    "subject.user_role",          # guest|user|admin
    "subject.account_status",     # active|locked|disabled
    "subject.mfa_verified",       # true|false

    # Device posture
    "device.managed",             # true|false
    "device.compliant",           # true|false
    "device.risk_level",          # low|medium|high

    # Environment signals
    "environment.ip_reputation",  # good|suspicious|malicious
    "environment.geo_risk",       # low|medium|high
    "environment.time_window",    # business_hours|after_hours

    # Resource sensitivity (optional condition)
    "resource.sensitivity",       # public|internal|restricted
}

class Condition(BaseModel):
    path: str
    op: Literal["eq", "neq", "in", "not_in", "gte_ranked", "lte_ranked"]
    value: Union[str, bool, int, List[Union[str, bool, int]]]

    @model_validator(mode="after")
    def _validate_path(self):
        if self.path not in ALLOWED_CONDITION_PATHS:
            raise ValueError(f"Condition path not allowed: {self.path}")
        return self

class Rule(BaseModel):
    rule_id: str = Field(..., min_length=3)
    effect: Literal["allow", "deny"]
    description: str = Field(..., min_length=5)

    # Target for website/login-style policies
    action: Literal["login", "view", "edit", "admin_panel"]
    app: str = Field(..., min_length=1)
    path_prefix: str = Field(..., min_length=1)
    sensitivity: Literal["public", "internal", "restricted"]

    # Conditions
    conditions_all: List[Condition] = Field(default_factory=list)  # AND
    conditions_any: List[Condition] = Field(default_factory=list)  # OR (optional)

    # Outputs/metadata
    obligations: List[str] = Field(default_factory=lambda: ["log"])
    nist_notes: List[str] = Field(default_factory=list)

class Example(BaseModel):
    name: str
    expect_allow: bool
    input: dict

class PolicyDraft(BaseModel):
    policy_id: str = Field(..., min_length=3)
    description: str = Field(..., min_length=5)
    app: str = Field(..., min_length=1)
    default_effect: Literal["deny", "allow"] = "deny"

    # Either provide explicit rules OR provide natural language for Gemini to draft
    rules: Optional[List[Rule]] = None
    nl_requirements: Optional[str] = None

    # Unit-test examples (allow/deny)
    examples: List[Example] = Field(default_factory=list)

    @model_validator(mode="after")
    def _rules_or_nl(self):
        if (not self.rules or len(self.rules) == 0) and (not self.nl_requirements or len(self.nl_requirements.strip()) == 0):
            raise ValueError("Provide either 'rules' (OPA-only) OR 'nl_requirements' (Gemini drafting).")
        return self

# For ranked comparisons
RANK = {"low": 0, "medium": 1, "high": 2}

rprint("[green]Schema loaded.[/green]")


Schema loaded.

**cell 6**

AML input ***(NEED edit this)***


YAML input options

OPA-only mode: Provide rules: explicitly (works without Gemini key).
Gemini drafting mode: Remove/empty rules: and provide nl_requirements:.

**cell 7**

for ***YAML*** we need the following info:

**real endpoints (*login, admin paths, API paths*)**

+

**the signals we want (*MFA, device posture, IP reputation, geo rules, business hours*)**

**Sample YAML (NEEDS edit)**

**V3**

In [ ]:
# === Cell 7: YAML Input UI (Done / Cancel) + parse + validate ===
# Requires: PolicyDraft already defined in earlier cells.

import yaml
from rich import print as rprint

# Widgets (Colab)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    !pip -q install ipywidgets
    import ipywidgets as widgets
    from IPython.display import display, clear_output

# Default YAML template (pre-filled so user can edit)
DEFAULT_YAML = """\
policy_id: website_login_v1
description: >
  NIST Zero Trust access control for a web app.
  Default deny. Require MFA + good device posture. Deny suspicious/malicious signals.
app: demo_webapp
default_effect: deny

rules:
  - rule_id: deny_login_bad_ip_or_geo
    effect: deny
    description: Deny login when IP is suspicious/malicious or geo risk is high.
    action: login
    app: demo_webapp
    path_prefix: /login
    sensitivity: internal
    conditions_any:
      - {path: environment.ip_reputation, op: in, value: [suspicious, malicious]}
      - {path: environment.geo_risk, op: eq, value: high}
    obligations: [log]
    nist_notes: ["continuous verification", "block high-risk signals"]

  - rule_id: deny_login_locked_or_disabled
    effect: deny
    description: Deny login if account is locked or disabled.
    action: login
    app: demo_webapp
    path_prefix: /login
    sensitivity: internal
    conditions_any:
      - {path: subject.account_status, op: eq, value: locked}
      - {path: subject.account_status, op: eq, value: disabled}
    obligations: [log]
    nist_notes: ["identity state gating"]

  - rule_id: allow_login_user_admin_strong
    effect: allow
    description: Allow login for active user/admin when MFA verified, device managed+compliant, and device risk <= medium.
    action: login
    app: demo_webapp
    path_prefix: /login
    sensitivity: internal
    conditions_all:
      - {path: subject.account_status, op: eq, value: active}
      - {path: subject.user_role, op: in, value: [user, admin]}
      - {path: subject.mfa_verified, op: eq, value: true}
      - {path: device.managed, op: eq, value: true}
      - {path: device.compliant, op: eq, value: true}
      - {path: device.risk_level, op: lte_ranked, value: medium}
    obligations: [log]
    nist_notes: ["least privilege", "continuous verification", "device posture check"]

  - rule_id: deny_admin_bad_signals
    effect: deny
    description: Deny admin panel if IP suspicious/malicious or geo risk high or device not compliant.
    action: admin_panel
    app: demo_webapp
    path_prefix: /admin
    sensitivity: restricted
    conditions_any:
      - {path: environment.ip_reputation, op: in, value: [suspicious, malicious]}
      - {path: environment.geo_risk, op: eq, value: high}
      - {path: device.compliant, op: eq, value: false}
    obligations: [log]
    nist_notes: ["privileged access hardening", "continuous verification"]

  - rule_id: deny_admin_without_mfa
    effect: deny
    description: Deny admin panel if MFA is not verified.
    action: admin_panel
    app: demo_webapp
    path_prefix: /admin
    sensitivity: restricted
    conditions_all:
      - {path: subject.mfa_verified, op: eq, value: false}
    obligations: [log]
    nist_notes: ["strong authentication for privileged access"]

  - rule_id: allow_admin_only_strict
    effect: allow
    description: Allow admin panel only for active admin with MFA verified and managed+compliant device and low risk.
    action: admin_panel
    app: demo_webapp
    path_prefix: /admin
    sensitivity: restricted
    conditions_all:
      - {path: subject.account_status, op: eq, value: active}
      - {path: subject.user_role, op: eq, value: admin}
      - {path: subject.mfa_verified, op: eq, value: true}
      - {path: device.managed, op: eq, value: true}
      - {path: device.compliant, op: eq, value: true}
      - {path: device.risk_level, op: eq, value: low}
    obligations: [log]
    nist_notes: ["least privilege", "step-up auth enforced", "admin isolation"]

examples:
  - name: login_good_user
    expect_allow: true
    input:
      subject: {user_role: user, account_status: active, mfa_verified: true}
      device: {managed: true, compliant: true, risk_level: medium}
      environment: {ip_reputation: good, geo_risk: low, time_window: business_hours}
      resource: {app: demo_webapp, path: /login, sensitivity: internal}
      action: {name: login}

  - name: login_bad_ip
    expect_allow: false
    input:
      subject: {user_role: user, account_status: active, mfa_verified: true}
      device: {managed: true, compliant: true, risk_level: low}
      environment: {ip_reputation: malicious, geo_risk: low, time_window: business_hours}
      resource: {app: demo_webapp, path: /login, sensitivity: internal}
      action: {name: login}

  - name: admin_good
    expect_allow: true
    input:
      subject: {user_role: admin, account_status: active, mfa_verified: true}
      device: {managed: true, compliant: true, risk_level: low}
      environment: {ip_reputation: good, geo_risk: low, time_window: after_hours}
      resource: {app: demo_webapp, path: /admin/settings, sensitivity: restricted}
      action: {name: admin_panel}
"""

# --- UI controls ---
yaml_box = widgets.Textarea(
    value=DEFAULT_YAML,
    description="YAML:",
    layout=widgets.Layout(width="100%", height="420px"),
)

done_btn = widgets.Button(description="Done", button_style="success")
cancel_btn = widgets.Button(description="Cancel", button_style="danger")

msg = widgets.Output()

# Global flags you can use later
YAML_CANCELLED = False
draft = None  # will be set only after successful validation

def show_ui():
    clear_output(wait=True)
    display(yaml_box, widgets.HBox([done_btn, cancel_btn]), msg)

def on_cancel(_):
    global YAML_CANCELLED, draft
    YAML_CANCELLED = True
    draft = None
    msg.clear_output()
    with msg:
        rprint("[yellow]Cancelled. YAML intake disabled; running is stopped for this step.[/yellow]")
        rprint("[yellow]If you want to re-enter YAML, re-run this cell.[/yellow]")

def on_done(_):
    global YAML_CANCELLED, draft
    YAML_CANCELLED = False
    text = (yaml_box.value or "").strip()

    msg.clear_output()
    if not text:
        with msg:
            rprint("[red]Error:[/red] YAML box is empty. Paste YAML then press Done.")
        show_ui()
        return

    try:
        raw = yaml.safe_load(text)
        # Validate against your PolicyDraft schema
        d = PolicyDraft(**raw)
        draft = d  # only set if validation succeeds
        with msg:
            rprint("[green]YAML parsed + validated successfully.[/green]")
            rprint(f"[cyan]policy_id:[/cyan] {draft.policy_id}")
            rprint(f"[cyan]rules:[/cyan] {len(draft.rules or [])}")
            rprint(f"[cyan]examples:[/cyan] {len(draft.examples)}")
            rprint("[green]You can now run the next cells.[/green]")
    except Exception as e:
        with msg:
            rprint("[red]Validation failed:[/red]")
            rprint(str(e))
            rprint("[yellow]Fix the YAML and press Done again.[/yellow]")
        show_ui()

done_btn.on_click(on_done)
cancel_btn.on_click(on_cancel)

# Show the UI initially
show_ui()

Textarea(value='policy_id: website_login_v1\ndescription: >\n  NIST Zero Trust access control for a web app.\n…

Output()

**NIST alignment check on YAML**

In [ ]:
def nist_alignment_report(draft: PolicyDraft) -> dict:
    report = {"pass": True, "checks": []}

    def add(check_name, ok, detail=""):
        report["checks"].append({"check": check_name, "ok": bool(ok), "detail": detail})
        if not ok:
            report["pass"] = False

    # 1) default deny
    add("Default deny", draft.default_effect == "deny",
        f"default_effect={draft.default_effect}")

    rules = draft.rules or []
    deny_rules = [r for r in rules if r.effect == "deny"]
    allow_rules = [r for r in rules if r.effect == "allow"]

    # 2) must have deny rules
    add("Has deny rules", len(deny_rules) > 0, f"deny_rules={len(deny_rules)}")

    # 3) must have allow rules
    add("Has allow rules", len(allow_rules) > 0, f"allow_rules={len(allow_rules)}")

    # 4) risky-signal deny coverage (ip_reputation or geo_risk)
    risky_paths = {"environment.ip_reputation", "environment.geo_risk"}
    risky_deny = False
    for r in deny_rules:
        used = {c.path for c in (r.conditions_all + r.conditions_any)}
        if used & risky_paths:
            risky_deny = True
            break
    add("Deny covers risky environment signals (ip/geo)", risky_deny,
        "Need a deny rule using environment.ip_reputation and/or environment.geo_risk")

    # Helper to check if a rule references a category of signals
    def has_any_path(rule, paths):
        used = {c.path for c in (rule.conditions_all + rule.conditions_any)}
        return bool(used & set(paths))

    ID_PATHS = {"subject.user_role", "subject.account_status", "subject.mfa_verified"}
    DEVICE_PATHS = {"device.managed", "device.compliant", "device.risk_level"}
    ENV_PATHS = {"environment.ip_reputation", "environment.geo_risk", "environment.time_window"}

    # 5) allow rules should include identity + device + environment checks (verify explicitly)
    ok_allow_signal_mix = True
    for r in allow_rules:
        # we require at least identity + (device OR environment). You can tighten this.
        ok = has_any_path(r, ID_PATHS) and (has_any_path(r, DEVICE_PATHS) or has_any_path(r, ENV_PATHS))
        if not ok:
            ok_allow_signal_mix = False
            break
    add("Allow rules use explicit verification signals (identity + device/env)", ok_allow_signal_mix,
        "Each allow rule should include identity + device/environment conditions")

    # 6) logging obligation present on allow rules
    log_ok = all(("log" in (r.obligations or [])) for r in allow_rules)
    add("Allow rules include log obligation", log_ok, "Add obligations: [log] to allow rules")

    # 7) admin_panel must require MFA (strong auth for privileged access)
    admin_allows = [r for r in allow_rules if r.action == "admin_panel"]
    if admin_allows:
        mfa_required = True
        for r in admin_allows:
            used = {c.path for c in (r.conditions_all + r.conditions_any)}
            if "subject.mfa_verified" not in used:
                mfa_required = False
                break
        add("Admin panel allow requires MFA", mfa_required,
            "Admin allow rules must include subject.mfa_verified == true")
    else:
        add("Has an admin_panel allow rule (optional)", True, "No admin_panel allow rule found (OK if not needed)")

    return report

rep = nist_alignment_report(draft)
print(json.dumps(rep, indent=2))
print("\nNIST alignment overall:", "PASS ✅" if rep["pass"] else "FAIL ❌")

{
  "pass": true,
  "checks": [
    {
      "check": "Default deny",
      "ok": true,
      "detail": "default_effect=deny"
    },
    {
      "check": "Has deny rules",
      "ok": true,
      "detail": "deny_rules=4"
    },
    {
      "check": "Has allow rules",
      "ok": true,
      "detail": "allow_rules=2"
    },
    {
      "check": "Deny covers risky environment signals (ip/geo)",
      "ok": true,
      "detail": "Need a deny rule using environment.ip_reputation and/or environment.geo_risk"
    },
    {
      "check": "Allow rules use explicit verification signals (identity + device/env)",
      "ok": true,
      "detail": "Each allow rule should include identity + device/environment conditions"
    },
    {
      "check": "Allow rules include log obligation",
      "ok": true,
      "detail": "Add obligations: [log] to allow rules"
    },
    {
      "check": "Admin panel allow requires MFA",
      "ok": true,
      "detail": "Admin allow rules must include subject.mfa_ver

**cell 8**

Optional Gemini drafting (only if rules missing)


If rules are missing/empty and you provided nl_requirements, then:

If Gemini key exists → Gemini drafts rules into the strict schema

If no key → you must provide rules (OPA-only mode)

**cell 9**

**Gemini drafting (strict JSON output)**

In [ ]:
def extract_json_obj(text: str) -> dict:
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m.group(0))

def build_gemini_prompt(d: PolicyDraft) -> str:
    schema = PolicyDraft.model_json_schema()
    return f"""
You draft NIST Zero Trust access-control rules for website/login authorization.
Return ONLY valid JSON that conforms to this schema (no markdown, no commentary):
{json.dumps(schema, indent=2)}

Hard constraints:
- default_effect must be "deny"
- Use ONLY these condition paths:
{sorted(ALLOWED_CONDITION_PATHS)}
- Include both deny and allow rules (deny for risky signals, allow for intended access)
- Actions limited to: login, view, edit, admin_panel
- Sensitivity limited to: public, internal, restricted

Input draft:
{d.model_dump_json(indent=2)}
""".strip()

def draft_rules_with_gemini(d: PolicyDraft) -> PolicyDraft:
    if not GEMINI_AVAILABLE:
        raise RuntimeError("Gemini not available.")

    from google import genai
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    client = genai.Client(api_key=api_key)

    prompt = build_gemini_prompt(d)
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    data = extract_json_obj(resp.text or "")
    return PolicyDraft(**data)

# Only draft if rules are missing/empty AND nl_requirements exists
if (not draft.rules or len(draft.rules) == 0) and draft.nl_requirements:
    if GEMINI_AVAILABLE:
        rprint("[cyan]Drafting rules with Gemini...[/cyan]")
        draft = draft_rules_with_gemini(draft)
        rprint("[green]Gemini draft validated.[/green]")
    else:
        raise RuntimeError("No rules provided and Gemini not available. Add explicit YAML rules or set GEMINI_API_KEY.")

rprint("[green]Final policy draft ready.[/green]")
draft


Final policy draft ready.

PolicyDraft(policy_id='website_login_v1', description='NIST Zero Trust access control for a web app. Default deny. Require MFA + good device posture. Deny suspicious/malicious signals.\n', app='demo_webapp', default_effect='deny', rules=[Rule(rule_id='deny_login_bad_ip_or_geo', effect='deny', description='Deny login when IP is suspicious/malicious or geo risk is high.', action='login', app='demo_webapp', path_prefix='/login', sensitivity='internal', conditions_all=[], conditions_any=[Condition(path='environment.ip_reputation', op='in', value=['suspicious', 'malicious']), Condition(path='environment.geo_risk', op='eq', value='high')], obligations=['log'], nist_notes=['continuous verification', 'block high-risk signals']), Rule(rule_id='deny_login_locked_or_disabled', effect='deny', description='Deny login if account is locked or disabled.', action='login', app='demo_webapp', path_prefix='/login', sensitivity='internal', conditions_all=[], conditions_any=[Condition(path='subject.account_

**cell 10**

Compile draft → Rego (printed output)


Now we compile into Rego:

default deny

deny overrides allow

exposes decision with matched_allow, matched_deny, obligations

We will print:

the draft JSON

the Rego policy code (copy/paste)

the Rego test code (copy/paste)

No persistent files are written.

**cell 11**

**Rego compiler**

***V4***

In [ ]:
def rego_lit(x):
    if isinstance(x, bool):
        return "true" if x else "false"
    if isinstance(x, (int, float)):
        return str(x)
    if isinstance(x, str):
        return json.dumps(x)
    if isinstance(x, list):
        return "[" + ", ".join(rego_lit(v) for v in x) + "]"
    if x is None:
        return "null"
    return json.dumps(x)

def _sanitize(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", name)

def compile_to_rego(d: PolicyDraft, package_name="zta.authz") -> str:
    if not d.rules or len(d.rules) == 0:
        raise ValueError("No rules available to compile.")

    lines = []
    lines.append(f"package {package_name}\n")
    lines.append('rank := {"low": 0, "medium": 1, "high": 2}\n')
    lines.append("default allow := false")
    lines.append("default deny := false\n")

    # Helper predicates (OR without using 'or')
    helper_blocks = []

    def cond_expr_or_helper(rule_id: str, idx: int, c: Condition) -> str:
        """
        Returns a safe Rego expression line.
        - For 'in' => create helper predicate with multiple bodies (OR)
        - For 'not_in' => helper + 'not helper'
        - Others => direct expression
        """
        path = "input." + c.path

        if c.op == "eq":
            return f"{path} == {rego_lit(c.value)}"
        if c.op == "neq":
            return f"{path} != {rego_lit(c.value)}"
        if c.op == "lte_ranked":
            return f"rank[{path}] <= rank[{rego_lit(c.value)}]"
        if c.op == "gte_ranked":
            return f"rank[{path}] >= rank[{rego_lit(c.value)}]"

        # Build helper for IN / NOT_IN to avoid 'or' expressions
        if c.op in ["in", "not_in"]:
            if not isinstance(c.value, list):
                raise ValueError(f"{c.op} requires list value")

            hname = f"cond_{_sanitize(rule_id)}_{idx}"
            # OR via multiple bodies
            for v in c.value:
                helper_blocks.append(f"{hname} if {{ {path} == {rego_lit(v)} }}")

            if c.op == "in":
                return f"{hname}"
            else:  # not_in
                return f"not {hname}"

        raise ValueError(f"Unsupported op: {c.op}")

    obligations_map_entries = []
    allow_blocks = []
    deny_blocks = []

    for r in d.rules:
        common = [
            f'input.action.name == "{r.action}"',
            f'input.resource.app == "{r.app}"',
            f'startswith(input.resource.path, "{r.path_prefix}")',
            f'input.resource.sensitivity == "{r.sensitivity}"'
        ]

        # AND conditions
        all_lines = []
        idx_counter = 0
        for x in common:
            all_lines.append("  " + x)

        for c in r.conditions_all:
            idx_counter += 1
            all_lines.append("  " + cond_expr_or_helper(r.rule_id, idx_counter, c))

        # OR conditions_any: implement as another helper any_<ruleid> with multiple bodies
        any_name = None
        if r.conditions_any:
            any_name = f"any_{_sanitize(r.rule_id)}"
            for c in r.conditions_any:
                idx_counter += 1
                expr = cond_expr_or_helper(r.rule_id, idx_counter, c)
                helper_blocks.append(f"{any_name} if {{ {expr} }}")
            all_lines.append(f"  {any_name}")

        head = "allow_rule" if r.effect == "allow" else "deny_rule"
        block = []
        block.append(f'{head}[{rego_lit(r.rule_id)}] if {{')  # Rego v1
        block.extend(all_lines if all_lines else ["  true"])
        block.append("}")

        if r.effect == "allow":
            allow_blocks.append("\n".join(block))
            obligations_map_entries.append(f"  {rego_lit(r.rule_id)}: {rego_lit(r.obligations)},")
        else:
            deny_blocks.append("\n".join(block))

    # Emit helper rules first
    if helper_blocks:
        lines.append("\n# --- Helper predicates (OR via multiple bodies) ---")
        lines.extend(helper_blocks)
        lines.append("")

    if allow_blocks:
        lines.append("# --- Allow rules ---")
        lines.extend(allow_blocks)
        lines.append("")
    if deny_blocks:
        lines.append("# --- Deny rules ---")
        lines.extend(deny_blocks)
        lines.append("")

    lines.append("matched_allow := {rid | allow_rule[rid]}")
    lines.append("matched_deny  := {rid | deny_rule[rid]}\n")

    lines.append("deny if { count(matched_deny) > 0 }")
    lines.append("allow if { count(matched_allow) > 0; not deny }\n")

    lines.append("rule_obligations := {")
    lines.extend(obligations_map_entries or ["  # no allow rules"])
    lines.append("}\n")

    lines.append('obligations := [o | some rid; allow_rule[rid]; o := rule_obligations[rid][_]]\n')

    lines.append('decision := {')
    lines.append('  "allow": allow,')
    lines.append('  "deny": deny,')
    lines.append('  "matched_allow": matched_allow,')
    lines.append('  "matched_deny": matched_deny,')
    lines.append('  "obligations": obligations')
    lines.append('}\n')

    return "\n".join(lines)

rego_code = compile_to_rego(draft, package_name="zta.authz")
rprint("[green]Rego compiled (no 'or' expressions).[/green]")

Rego compiled (no 'or' expressions).

**NIST alignment checks on Rego (static + behavioral)**

In [ ]:
import re, json, tempfile, subprocess, os

# Set True if you want notebook to stop on NIST FAIL; False = just warn and continue
STRICT_NIST_GATE = False

def nist_rego_static_checks(rego_code: str) -> dict:
    checks = []
    def add(name, ok, detail=""):
        checks.append({"check": name, "ok": bool(ok), "detail": detail})

    code = rego_code or ""

    # 1) Deny-by-default posture (core)
    add(
        "Has default allow := false (deny-by-default posture)",
        ("default allow := false" in code),
        "Expected exact line: default allow := false"
    )

    # 2) deny overrides allow logic
    add(
        "Allow depends on not deny (deny overrides allow)",
        ("allow if {" in code and "not deny" in code),
        "Expected allow rule to include `not deny`"
    )

    # 3) Decision object exists
    add(
        "Decision object exported",
        ('decision := {' in code and '"allow": allow' in code),
        "Expected `decision := { ... }` with allow/deny fields"
    )

    # 4) Signal usage (Zero Trust context signals)
    add(
        "Uses identity signals (subject)",
        "input.subject." in code,
        "Expected subject attributes such as role/account_status/mfa_verified"
    )
    add(
        "Uses device posture signals",
        "input.device." in code,
        "Expected device attributes such as managed/compliant/risk_level"
    )
    add(
        "Uses environment signals",
        "input.environment." in code,
        "Expected env attributes such as ip_reputation/geo_risk/time_window"
    )

    # 5) Resource scope checks
    add(
        "Has login path scope",
        'startswith(input.resource.path, "/login")' in code,
        "Expected login path scope rule"
    )
    add(
        "Has admin path scope",
        'startswith(input.resource.path, "/admin")' in code,
        "Expected admin path scope rule"
    )

    # 6) Obligations/logging mechanism present
    add(
        "Has obligations support",
        ("rule_obligations :=" in code or "rule_obligations :=" in code.replace(" ", "")) or ("obligations :=" in code),
        "Expected rule_obligations/obligations mapping in generated policy"
    )

    # 7) Risk deny coverage (tolerant heuristic)
    # Instead of requiring ip/geo inside deny_rule block (which may use helpers),
    # require deny rules to exist + risky signals referenced somewhere in code.
    has_deny_rules = "deny_rule[" in code
    has_risky_signals = ("input.environment.ip_reputation" in code) or ("input.environment.geo_risk" in code)
    add(
        "Has deny rules and risky-signal logic",
        has_deny_rules and has_risky_signals,
        "Expected deny_rule[...] plus ip_reputation and/or geo_risk checks (directly or via helper predicates)"
    )

    passed = all(c["ok"] for c in checks)
    return {"pass": passed, "checks": checks}

def opa_eval_decision_temp(rego_code: str, input_obj: dict, package_name="zta.authz") -> dict:
    with tempfile.TemporaryDirectory() as td:
        with open(os.path.join(td, "policy.rego"), "w", encoding="utf-8") as f:
            f.write(rego_code)
        in_path = os.path.join(td, "input.json")
        with open(in_path, "w", encoding="utf-8") as f:
            f.write(json.dumps(input_obj))

        q = f"data.{package_name}.decision"
        p = subprocess.run(
            ["opa", "eval", "-d", td, "-i", in_path, q, "--format=json"],
            capture_output=True, text=True
        )
        if p.returncode != 0:
            raise RuntimeError(
                "OPA eval failed in NIST behavior checks\n"
                f"Query: {q}\nSTDERR:\n{p.stderr}\nSTDOUT:\n{p.stdout}"
            )

        out = json.loads(p.stdout)
        return out["result"][0]["expressions"][0]["value"]

def nist_rego_behavior_checks(rego_code: str, package_name="zta.authz") -> dict:
    checks = []
    def add(name, ok, detail=""):
        checks.append({"check": name, "ok": bool(ok), "detail": detail})

    try:
        # 1) Allow good login (baseline expected behavior)
        good_login = {
            "subject": {"user_role":"user","account_status":"active","mfa_verified": True},
            "device": {"managed": True, "compliant": True, "risk_level": "medium"},
            "environment": {"ip_reputation":"good","geo_risk":"low","time_window":"business_hours"},
            "resource": {"app":"demo_webapp","path":"/login","sensitivity":"internal"},
            "action": {"name":"login"}
        }
        dec = opa_eval_decision_temp(rego_code, good_login, package_name)
        add("Allow good login", dec.get("allow") is True, f"decision={dec}")

        # 2) Deny malicious IP login
        bad_ip = {
            **good_login,
            "environment": {**good_login["environment"], "ip_reputation":"malicious"}
        }
        dec = opa_eval_decision_temp(rego_code, bad_ip, package_name)
        add("Deny malicious IP login", dec.get("allow") is False, f"decision={dec}")

        # 3) Deny admin panel without MFA
        admin_no_mfa = {
            "subject": {"user_role":"admin","account_status":"active","mfa_verified": False},
            "device": {"managed": True, "compliant": True, "risk_level": "low"},
            "environment": {"ip_reputation":"good","geo_risk":"low","time_window":"after_hours"},
            "resource": {"app":"demo_webapp","path":"/admin/settings","sensitivity":"restricted"},
            "action": {"name":"admin_panel"}
        }
        dec = opa_eval_decision_temp(rego_code, admin_no_mfa, package_name)
        add("Deny admin_panel without MFA", dec.get("allow") is False, f"decision={dec}")

    except Exception as e:
        add("Behavior checks execution", False, str(e))

    passed = all(c["ok"] for c in checks)
    return {"pass": passed, "checks": checks}

def print_nist_report(title, rep):
    print("\n" + "="*12 + f" {title} " + "="*12)
    print("PASS ✅" if rep["pass"] else "FAIL ❌")
    for c in rep["checks"]:
        mark = "✅" if c["ok"] else "❌"
        print(f"{mark} {c['check']}")
        if c.get("detail"):
            print(f"   ↳ {c['detail']}")

def failed_checks(rep):
    return [c for c in rep["checks"] if not c["ok"]]

# --- Run checks ---
static_rep = nist_rego_static_checks(rego_code)
beh_rep = nist_rego_behavior_checks(rego_code, package_name="zta.authz")

print_nist_report("NIST ALIGNMENT (REGO) - STATIC", static_rep)
print_nist_report("NIST ALIGNMENT (REGO) - BEHAVIOR", beh_rep)

overall_pass = static_rep["pass"] and beh_rep["pass"]
print("\nNIST alignment overall:", "PASS ✅" if overall_pass else "FAIL ❌")

if not overall_pass:
    print("\nFailed static checks:")
    for c in failed_checks(static_rep):
        print("-", c["check"], "|", c.get("detail",""))
    print("\nFailed behavior checks:")
    for c in failed_checks(beh_rep):
        print("-", c["check"], "|", c.get("detail",""))

    if STRICT_NIST_GATE:
        raise RuntimeError("NIST alignment checks on Rego FAILED. Fix YAML/rules or Gemini output before proceeding.")
    else:
        print("\n[WARN] STRICT_NIST_GATE=False, so notebook will continue.")


============ NIST ALIGNMENT (REGO) - STATIC ============
PASS ✅
✅ Has default allow := false (deny-by-default posture)
   ↳ Expected exact line: default allow := false
✅ Allow depends on not deny (deny overrides allow)
   ↳ Expected allow rule to include `not deny`
✅ Decision object exported
   ↳ Expected `decision := { ... }` with allow/deny fields
✅ Uses identity signals (subject)
   ↳ Expected subject attributes such as role/account_status/mfa_verified
✅ Uses device posture signals
   ↳ Expected device attributes such as managed/compliant/risk_level
✅ Uses environment signals
   ↳ Expected env attributes such as ip_reputation/geo_risk/time_window
✅ Has login path scope
   ↳ Expected login path scope rule
✅ Has admin path scope
   ↳ Expected admin path scope rule
✅ Has obligations support
   ↳ Expected rule_obligations/obligations mapping in generated policy
✅ Has deny rules and risky-signal logic
   ↳ Expected deny_rule[...] plus ip_reputation and/or geo_risk checks (directly or vi

**cell 12**

Print draft JSON + Rego (copy/paste)


This prints:

policy draft JSON

Rego policy code (copy/paste)

If the output is long, Colab may collapse it; we also provide a scrollable viewer.

**cell 13**

**Print draft JSON + Rego**

In [ ]:
print("\n" + "="*12 + " POLICY DRAFT JSON (AUDITABLE) " + "="*12 + "\n")
print(draft.model_dump_json(indent=2))

print("\n" + "="*12 + " GENERATED REGO POLICY (COPY/PASTE) " + "="*12 + "\n")
print(rego_code)
print("\n" + "="*60 + "\n")



============ POLICY DRAFT JSON (AUDITABLE) ============

{
  "policy_id": "website_login_v1",
  "description": "NIST Zero Trust access control for a web app. Default deny. Require MFA + good device posture. Deny suspicious/malicious signals.\n",
  "app": "demo_webapp",
  "default_effect": "deny",
  "rules": [
    {
      "rule_id": "deny_login_bad_ip_or_geo",
      "effect": "deny",
      "description": "Deny login when IP is suspicious/malicious or geo risk is high.",
      "action": "login",
      "app": "demo_webapp",
      "path_prefix": "/login",
      "sensitivity": "internal",
      "conditions_all": [],
      "conditions_any": [
        {
          "path": "environment.ip_reputation",
          "op": "in",
          "value": [
            "suspicious",
            "malicious"
          ]
        },
        {
          "path": "environment.geo_risk",
          "op": "eq",
          "value": "high"
        }
      ],
      "obligations": [
        "log"
      ],
      "nist_note

**cell 14**

Optional: scrollable Rego viewer (still copy/paste)


If the Rego is big and Colab collapses the output, we need to use this scrollable box.

**cell 15**

**Scrollable viewer**

In [ ]:
from IPython.display import display, HTML

escaped = (rego_code
           .replace("&","&amp;")
           .replace("<","&lt;")
           .replace(">","&gt;"))

html = f"""
<div style="white-space:pre; font-family:monospace; font-size:12px;
            border:1px solid #ddd; padding:10px; max-height:520px; overflow:auto;">
{escaped}
</div>
"""
display(HTML(html))


**cell 16**

Generate Rego unit tests (printed output)


We convert our YAML examples into a Rego test module so OPA can validate allow/deny behavior.

**cell 17**

**Generate + print Rego tests**

**GENERATOR V2**

In [ ]:
def generate_rego_tests(d: PolicyDraft, package_name="zta.authz") -> str:
    lines = []
    lines.append(f"package {package_name}\n")

    for ex in d.examples:
        test_name = re.sub(r"[^a-zA-Z0-9_]", "_", ex.name)
        expect = "true" if ex.expect_allow else "false"
        inp = json.dumps(ex.input, indent=2)

        # ✅ Rego v1 test rule syntax
        lines.append(f"test_{test_name} if {{")
        lines.append(f"  dec := data.{package_name}.decision with input as {inp}")
        lines.append(f"  dec.allow == {expect}")
        lines.append("}\n")

    return "\n".join(lines)

tests_code = generate_rego_tests(draft, package_name="zta.authz")

print("\n" + "="*12 + " GENERATED REGO TESTS (COPY/PASTE) " + "="*12 + "\n")
print(tests_code)
print("\n" + "="*60 + "\n")


============ GENERATED REGO TESTS (COPY/PASTE) ============

package zta.authz

test_login_good_user if {
  dec := data.zta.authz.decision with input as {
  "subject": {
    "user_role": "user",
    "account_status": "active",
    "mfa_verified": true
  },
  "device": {
    "managed": true,
    "compliant": true,
    "risk_level": "medium"
  },
  "environment": {
    "ip_reputation": "good",
    "geo_risk": "low",
    "time_window": "business_hours"
  },
  "resource": {
    "app": "demo_webapp",
    "path": "/login",
    "sensitivity": "internal"
  },
  "action": {
    "name": "login"
  }
}
  dec.allow == true
}

test_login_bad_ip if {
  dec := data.zta.authz.decision with input as {
  "subject": {
    "user_role": "user",
    "account_status": "active",
    "mfa_verified": true
  },
  "device": {
    "managed": true,
    "compliant": true,
    "risk_level": "low"
  },
  "environment": {
    "ip_reputation": "malicious",
    "geo_risk": "low",
    "time_window": "business_hours"
  },


**cell 18**

Run OPA check/test/eval without saving files


OPA needs policy/test files, so we:

write them into a temporary directory

run opa check, opa test, opa eval

print results

the temp directory is deleted automatically → nothing is saved.

**cell 19**

**OPA check/test/eval using temporary files (auto-deleted)**

**V2**, it only works with **V2** of **Cell 21**

In [ ]:
import os, json, tempfile, subprocess

def opa_check_test_eval(rego_policy: str, rego_tests: str, examples, package_name="zta.authz"):
    with tempfile.TemporaryDirectory() as td:
        policy_path = os.path.join(td, "policy.rego")
        test_path = os.path.join(td, "policy_test.rego")

        # Write temporary files (auto-deleted)
        with open(policy_path, "w", encoding="utf-8") as f:
            f.write(rego_policy)
        with open(test_path, "w", encoding="utf-8") as f:
            f.write(rego_tests)

        # 1) OPA check
        p = subprocess.run(["opa", "check", td], capture_output=True, text=True)
        print("\n" + "="*12 + " OPA CHECK OUTPUT " + "="*12 + "\n")
        print(p.stdout.strip() or "(no stdout)")
        if p.returncode != 0:
            print(p.stderr.strip() or "(no stderr)")
            raise RuntimeError("OPA check failed")

        # 2) OPA test
        p = subprocess.run(["opa", "test", td, "-v"], capture_output=True, text=True)
        print("\n" + "="*12 + " OPA TEST OUTPUT " + "="*12 + "\n")
        print((p.stdout.strip() or "(no stdout)")[:4000])
        if p.returncode != 0:
            print(p.stderr.strip() or "(no stderr)")
            raise RuntimeError("OPA tests failed")

        # 3) OPA eval each example input
        print("\n" + "="*12 + " OPA EVAL RESULTS (EACH EXAMPLE) " + "="*12 + "\n")
        q = f"data.{package_name}.decision"

        for ex in examples:
            in_path = os.path.join(td, "input.json")
            with open(in_path, "w", encoding="utf-8") as f:
                f.write(json.dumps(ex.input))

            # ✅ IMPORTANT: use -i (lowercase) for input file
            p = subprocess.run(
                ["opa", "eval", "-d", td, "-i", in_path, q, "--format=json"],
                capture_output=True,
                text=True
            )

            if p.returncode != 0:
                print(f"- {ex.name}: OPA eval FAILED")
                print("  STDOUT:", p.stdout.strip() or "(empty)")
                print("  STDERR:", p.stderr.strip() or "(empty)")
                continue

            out = json.loads(p.stdout)
            decision = out["result"][0]["expressions"][0]["value"]
            actual_allow = bool(decision.get("allow"))
            status = "PASS" if actual_allow == ex.expect_allow else "FAIL"

            print(f"- {ex.name}: expected_allow={ex.expect_allow} | actual_allow={actual_allow} | {status}")
            print(f"  matched_allow={sorted(list(decision.get('matched_allow', [])))}")
            print(f"  matched_deny={sorted(list(decision.get('matched_deny', [])))}")
            print(f"  obligations={decision.get('obligations', [])}\n")

# Run it
opa_check_test_eval(rego_code, tests_code, draft.examples, package_name="zta.authz")
rprint("[green]OPA run complete.[/green]")


============ OPA CHECK OUTPUT ============

(no stdout)

============ OPA TEST OUTPUT ============

/tmp/tmpdt37jpwg/policy_test.rego:
data.zta.authz.test_admin_good: PASS (1.41031ms)
data.zta.authz.test_login_good_user: PASS (1.62201ms)
data.zta.authz.test_login_bad_ip: PASS (1.48362ms)
--------------------------------------------------------------------------------
PASS: 3/3

============ OPA EVAL RESULTS (EACH EXAMPLE) ============

- login_good_user: expected_allow=True | actual_allow=True | PASS
  matched_allow=['allow_login_user_admin_strong']
  matched_deny=[]
  obligations=['log']

- login_bad_ip: expected_allow=False | actual_allow=False | PASS
  matched_allow=['allow_login_user_admin_strong']
  matched_deny=['deny_login_bad_ip_or_geo']
  obligations=['log']

- admin_good: expected_allow=True | actual_allow=True | PASS
  matched_allow=['allow_admin_only_strict']
  matched_deny=[]
  obligations=['log']



OPA run complete.

**Cell 20**

Evaluate your own custom request (printed)

Edit the input JSON below and run the cell to see the full decision output.

**Cell 21**

**Custom eval input (prints decision)**

**V2**

In [ ]:
import json, os, tempfile, subprocess

custom_input = {
    "subject": {"user_role": "user", "account_status": "active", "mfa_verified": True},
    "device": {"managed": True, "compliant": True, "risk_level": "medium"},
    "environment": {"ip_reputation": "good", "geo_risk": "low", "time_window": "business_hours"},
    "resource": {"app": "demo_webapp", "path": "/login", "sensitivity": "internal"},
    "action": {"name": "login"}
}

with tempfile.TemporaryDirectory() as td:
    # write policy
    policy_path = os.path.join(td, "policy.rego")
    with open(policy_path, "w", encoding="utf-8") as f:
        f.write(rego_code)

    # write input
    in_path = os.path.join(td, "input.json")
    with open(in_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(custom_input))

    q = "data.zta.authz.decision"

    # ✅ IMPORTANT: use -i (lowercase), not -I
    p = subprocess.run(
        ["opa", "eval", "-d", td, "-i", in_path, q, "--format=json"],
        capture_output=True,
        text=True
    )

    if p.returncode != 0:
        print("\nOPA EVAL FAILED")
        print("STDOUT:\n", p.stdout)
        print("STDERR:\n", p.stderr)
        raise RuntimeError("opa eval failed")

    out = json.loads(p.stdout)
    decision = out["result"][0]["expressions"][0]["value"]

print("\n" + "="*12 + " CUSTOM INPUT " + "="*12 + "\n")
print(json.dumps(custom_input, indent=2))
print("\n" + "="*12 + " OPA DECISION OUTPUT " + "="*12 + "\n")
print(json.dumps(decision, indent=2))


============ CUSTOM INPUT ============

{
  "subject": {
    "user_role": "user",
    "account_status": "active",
    "mfa_verified": true
  },
  "device": {
    "managed": true,
    "compliant": true,
    "risk_level": "medium"
  },
  "environment": {
    "ip_reputation": "good",
    "geo_risk": "low",
    "time_window": "business_hours"
  },
  "resource": {
    "app": "demo_webapp",
    "path": "/login",
    "sensitivity": "internal"
  },
  "action": {
    "name": "login"
  }
}

============ OPA DECISION OUTPUT ============

{
  "allow": true,
  "deny": false,
  "matched_allow": [
    "allow_login_user_admin_strong"
  ],
  "matched_deny": [],
  "obligations": [
    "log"
  ]
}


Simulation Cell (Shadow Mode) ✅

This generates a bunch of synthetic access requests, evaluates them using OPA, and prints metrics like allow rate, deny reasons, matched rules.

Simulation Cell:

Shadow-mode simulation: generate many login/admin requests (synthetic), evaluate with OPA, summarize allow/deny rates and matched rules. This is NOT a unit test; it’s behavior analysis.

**Simulation Cell (Code)**

**V2**

In [ ]:
import os, json, tempfile, subprocess
from collections import Counter
import random

def opa_eval_with_temp(policy_rego: str, ctx: dict, package_name="zta.authz") -> dict:
    with tempfile.TemporaryDirectory() as td:
        # write policy
        with open(os.path.join(td, "policy.rego"), "w", encoding="utf-8") as f:
            f.write(policy_rego)

        # write input
        in_path = os.path.join(td, "input.json")
        with open(in_path, "w", encoding="utf-8") as f:
            f.write(json.dumps(ctx))

        q = f"data.{package_name}.decision"

        # ✅ IMPORTANT: -i (lowercase)
        p = subprocess.run(
            ["opa", "eval", "-d", td, "-i", in_path, q, "--format=json"],
            capture_output=True,
            text=True
        )

        if p.returncode != 0:
            print("\nOPA eval FAILED during simulation")
            print("Query:", q)
            print("STDOUT:\n", p.stdout.strip() or "(empty)")
            print("STDERR:\n", p.stderr.strip() or "(empty)")
            print("Input was:\n", json.dumps(ctx, indent=2))
            raise RuntimeError("opa eval failed in simulation")

        out = json.loads(p.stdout)
        return out["result"][0]["expressions"][0]["value"]

# --- Simulation generator (keep your own if you already have one) ---
def random_ctx():
    action = random.choice(["login", "admin_panel"])
    path = "/login" if action == "login" else random.choice(["/admin", "/admin/settings", "/admin/users"])

    ctx = {
        "subject": {
            "user_role": random.choice(["guest","user","admin"]),
            "account_status": random.choice(["active","active","active","locked"]),
            "mfa_verified": random.choice([True, False]),
        },
        "device": {
            "managed": random.choice([True, False]),
            "compliant": random.choice([True, False]),
            "risk_level": random.choice(["low","medium","high"]),
        },
        "environment": {
            "ip_reputation": random.choice(["good","good","suspicious","malicious"]),
            "geo_risk": random.choice(["low","medium","high"]),
            "time_window": random.choice(["business_hours","after_hours"]),
        },
        "resource": {
            "app": "demo_webapp",
            "path": path,
            "sensitivity": "internal" if action=="login" else "restricted"
        },
        "action": {"name": action},
    }
    return ctx

# --- Run shadow-mode simulation ---
N = 5000
decisions = []
allow_rules = Counter()
deny_rules = Counter()

for _ in range(N):
    ctx = random_ctx()
    dec = opa_eval_with_temp(rego_code, ctx, package_name="zta.authz")
    decisions.append(bool(dec.get("allow", False)))
    for rid in dec.get("matched_allow", []):
        allow_rules[rid] += 1
    for rid in dec.get("matched_deny", []):
        deny_rules[rid] += 1

allow_rate = sum(decisions) / N
print("\n" + "="*12 + " SHADOW MODE SIMULATION SUMMARY " + "="*12)
print("Total requests:", N)
print("Allow rate:", round(allow_rate, 3))

print("\nTop allow rules:")
for k,v in allow_rules.most_common(10):
    print(f"  {k}: {v}")

print("\nTop deny rules:")
for k,v in deny_rules.most_common(10):
    print(f"  {k}: {v}")
print("="*60)


============ SHADOW MODE SIMULATION SUMMARY ============
Total requests: 5000
Allow rate: 0.008

Top allow rules:
  allow_login_user_admin_strong: 113
  allow_admin_only_strict: 26

Top deny rules:
  deny_admin_bad_signals: 2128
  deny_login_bad_ip_or_geo: 1600
  deny_admin_without_mfa: 1313
  deny_login_locked_or_disabled: 606


What we can copy/paste from outputs

Rego policy code is printed in Cell 13 (and scrollable in Cell 15) ✅

Rego tests printed in Cell 17 ✅

OPA results printed in Cell 19 ✅

Draft JSON printed in Cell 13 ✅

to write YAML we need:

endpoints (login, admin paths, API paths)
+
the signals we want (MFA, device posture, IP reputation, geo rules, business hours),

here in this code have:

Unit tests (golden tests): examples → opa test

Simulation: random dataset → allow/deny stats

***NOTE FROM AI:***

1) Unit tests (golden tests) ✅

Purpose: “Given a known input, the policy must allow/deny exactly as expected.”

Where it is in the notebook:

Cell 16–17: generate_rego_tests(...) creates Rego test rules from your YAML examples.

Cell 19: opa test runs those tests.

Exactly what makes it a test:

Each YAML example has expect_allow: true/false

The generated policy_test.rego contains test_<name> { ... dec.allow == true/false }

OPA returns PASS/FAIL.

So in your design language:

Golden tests = the YAML examples + generate_rego_tests() + opa test.

2) Static validation checks ✅

Purpose: catch mistakes before running anything (schema errors, unsupported attributes, malformed rules).

Where it is:

Cell 7: YAML is parsed and validated via PolicyDraft(**raw)

Cell 11: compilation step (will fail if the draft is missing rules)

Cell 19: opa check validates Rego syntax/types.

This is more like:

“schema validation / static validator” from your architecture.

Cell 19 does something close to simulation, but only over the examples (which are actually tests).

Cell 21 is a “single request evaluation” (a demo), not a real simulation dataset.

Quick mapping summary

✅ Tests

Cell 16–17: generate tests from examples

Cell 19: opa test (PASS/FAIL)

✅ Simulation (Shadow mode)

Added in the last part

✅ Simple evaluation demo

Cell 21: evaluates one custom input (not a test, not simulation)